# AISO Few-Shot Example Selector

## 한 줄 요약
> **AISO = 풀(Pool) 안에서 서로 겹치지 않는 알짜배기 예시 조합을 찾아주는 다양성 확보 샘플러**

---

## 왜 Few-Shot에 쓰는가?

LLM에 넘기는 Few-shot 예시들이 비슷한 구조로만 구성되면 모델이 그 패턴에 과적합됩니다.  
예를 들어 NER 태깅 태스크에서:
- 전부 짧고 단순한 문장만 뽑으면 → 긴 복잡한 문장에서 성능 저하
- 전부 인명(PER) 위주면 → 기관명(ORG), 지명(LOC) 인식 편향

**AISO는 에이전트들이 서로 밀어내면서 다른 구조적 역할을 맡도록 강제합니다.**  
결과적으로 선택된 예시 세트는 다양한 패턴을 커버합니다.

---

## 핵심 조건 (이게 없으면 AISO가 랜덤과 같아짐)

1. **도메인 피처** — 예시들의 구조적 차이를 숫자로 표현한 벡터
2. **스코어 함수** — 선택된 세트의 품질 평가 (외부 비교용; AISO 내부 루프에서 직접 쓰지 않음)

피처가 잘 설계되면 AISO가 작동합니다. 피처가 PCA처럼 뭉개진 형태면 → 랜덤과 동일.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from itertools import combinations
import re

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

---
## 1. 알고리즘 직관

### 상태 변수
| 기호 | 의미 |
|------|------|
| $X_i \in [0,1]^D$ | 에이전트 $i$의 피처 공간 내 위치 |
| $W_i \in \Delta^{K-1}$ | 에이전트 $i$의 타입 벡터 (확률 심플렉스) |
| $M \in \mathbb{R}^{K \times K}$ | 호환성 행렬 (고정, 랜덤 초기화) |

### 핵심 메커니즘

**호환성**: $c_{ij} = W_i^\top M W_j$
- 양수 → 에이전트 $i$가 $j$ 쪽으로 끌림
- 음수 → 에이전트 $i$가 $j$로부터 밀려남

**위치 업데이트**: 인접 풀 아이템으로 스냅
```
force = attraction_force + repulsion_force
candidate = X_i + α × force
X_i ← nearest pool item to candidate
visit[nearest] += 1
```

**타입 업데이트** (이동 성공 시): $W_i \leftarrow \text{normalize}((1-\beta)W_i + \beta W_j)$

→ 자주 상호작용하는 에이전트들이 유사한 타입으로 수렴 = 소프트 클러스터 형성  
→ 호환되지 않는 에이전트들은 서로 다른 피처 공간 영역을 탐색

**선택**: `visit` 빈도에 비례해 최종 예시 샘플링

---
## 2. 예시 데이터: NER 태깅 Few-Shot 풀

40개 문장으로 구성된 풀. 각 문장은 개체명 어노테이션을 포함합니다.  
실제 사용 시 이 부분을 본인의 풀로 교체하세요.

In [ ]:
# 풀 정의: (text, entities) 쌍
# entities = [(span, type), ...] — PER: 인명, ORG: 기관, LOC: 지명, DATE: 날짜, MISC: 기타

POOL = [
    # --- 짧고 단순, 인명 중심 ---
    {"text": "Steve Jobs founded Apple.",
     "entities": [("Steve Jobs", "PER"), ("Apple", "ORG")]},
    {"text": "Marie Curie won the Nobel Prize.",
     "entities": [("Marie Curie", "PER"), ("Nobel Prize", "MISC")]},
    {"text": "Elon Musk leads Tesla.",
     "entities": [("Elon Musk", "PER"), ("Tesla", "ORG")]},
    {"text": "Barack Obama visited Berlin.",
     "entities": [("Barack Obama", "PER"), ("Berlin", "LOC")]},
    {"text": "Ada Lovelace was a mathematician.",
     "entities": [("Ada Lovelace", "PER")]},

    # --- 기관 중심 ---
    {"text": "Google LLC was incorporated in Delaware.",
     "entities": [("Google LLC", "ORG"), ("Delaware", "LOC")]},
    {"text": "The United Nations held a summit in New York.",
     "entities": [("United Nations", "ORG"), ("New York", "LOC")]},
    {"text": "Harvard University announced new funding.",
     "entities": [("Harvard University", "ORG")]},
    {"text": "NASA launched the James Webb Space Telescope.",
     "entities": [("NASA", "ORG"), ("James Webb Space Telescope", "MISC")]},
    {"text": "The European Central Bank raised interest rates.",
     "entities": [("European Central Bank", "ORG")]},

    # --- 지명 중심 ---
    {"text": "Tokyo hosted the 2020 Summer Olympics.",
     "entities": [("Tokyo", "LOC"), ("2020 Summer Olympics", "MISC")]},
    {"text": "The Amazon River flows through Brazil.",
     "entities": [("Amazon River", "LOC"), ("Brazil", "LOC")]},
    {"text": "Mount Everest is located in Nepal.",
     "entities": [("Mount Everest", "LOC"), ("Nepal", "LOC")]},
    {"text": "Paris is the capital of France.",
     "entities": [("Paris", "LOC"), ("France", "LOC")]},
    {"text": "The Great Wall of China spans thousands of kilometers.",
     "entities": [("Great Wall of China", "LOC")]},

    # --- 날짜/시간 중심 ---
    {"text": "On July 20, 1969, Neil Armstrong landed on the Moon.",
     "entities": [("July 20, 1969", "DATE"), ("Neil Armstrong", "PER"), ("Moon", "LOC")]},
    {"text": "World War II ended in 1945.",
     "entities": [("World War II", "MISC"), ("1945", "DATE")]},
    {"text": "The iPhone was first released in June 2007.",
     "entities": [("iPhone", "MISC"), ("June 2007", "DATE")]},
    {"text": "In 2024, the global AI market exceeded $500 billion.",
     "entities": [("2024", "DATE")]},
    {"text": "The treaty was signed on March 15, 1848, in Guadalupe.",
     "entities": [("March 15, 1848", "DATE"), ("Guadalupe", "LOC")]},

    # --- 복잡한 문장, 다수 개체 ---
    {"text": "In 2018, Jeff Bezos of Amazon acquired Whole Foods in Seattle for $13.7 billion.",
     "entities": [("2018", "DATE"), ("Jeff Bezos", "PER"), ("Amazon", "ORG"), ("Whole Foods", "ORG"), ("Seattle", "LOC")]},
    {"text": "Angela Merkel, Chancellor of Germany from 2005 to 2021, met Emmanuel Macron in Paris.",
     "entities": [("Angela Merkel", "PER"), ("Germany", "LOC"), ("2005", "DATE"), ("2021", "DATE"), ("Emmanuel Macron", "PER"), ("Paris", "LOC")]},
    {"text": "Tesla, headquartered in Austin, Texas, reported Q3 2023 earnings of $23.35 billion.",
     "entities": [("Tesla", "ORG"), ("Austin", "LOC"), ("Texas", "LOC"), ("Q3 2023", "DATE")]},
    {"text": "The WHO and CDC jointly announced on January 5, 2020 that a novel coronavirus had been detected in Wuhan.",
     "entities": [("WHO", "ORG"), ("CDC", "ORG"), ("January 5, 2020", "DATE"), ("Wuhan", "LOC")]},
    {"text": "SpaceX, founded by Elon Musk in 2002 in Hawthorne, California, launched its Falcon 9 rocket.",
     "entities": [("SpaceX", "ORG"), ("Elon Musk", "PER"), ("2002", "DATE"), ("Hawthorne", "LOC"), ("California", "LOC"), ("Falcon 9", "MISC")]},

    # --- 개체명 모호성 / 경계 어려운 케이스 ---
    {"text": "Apple's quarterly earnings beat Wall Street expectations.",
     "entities": [("Apple", "ORG"), ("Wall Street", "LOC")]},
    {"text": "New York Times reported that the Fed raised rates.",
     "entities": [("New York Times", "ORG"), ("Fed", "ORG")]},
    {"text": "Jordan won the championship three times consecutively.",
     "entities": [("Jordan", "PER")]},  # 모호: 인명 vs 지명
    {"text": "The Amazon acquisition of MGM studios was approved by the FTC.",
     "entities": [("Amazon", "ORG"), ("MGM", "ORG"), ("FTC", "ORG")]},
    {"text": "Microsoft Office 365, launched in 2011, serves over 300 million users.",
     "entities": [("Microsoft Office 365", "MISC"), ("2011", "DATE")]},

    # --- 긴 복잡 문장 ---
    {"text": "The International Monetary Fund, based in Washington D.C. and led by Kristalina Georgieva since October 2019, released its annual economic outlook.",
     "entities": [("International Monetary Fund", "ORG"), ("Washington D.C.", "LOC"), ("Kristalina Georgieva", "PER"), ("October 2019", "DATE")]},
    {"text": "Researchers at MIT, Stanford, and the University of Cambridge published findings suggesting that GPT-4 outperforms human experts on the bar exam.",
     "entities": [("MIT", "ORG"), ("Stanford", "ORG"), ("University of Cambridge", "ORG"), ("GPT-4", "MISC")]},
    {"text": "During the G7 summit held in Hiroshima, Japan in May 2023, leaders from the United States, United Kingdom, France, Germany, Italy, Canada, and Japan discussed global security.",
     "entities": [("G7", "MISC"), ("Hiroshima", "LOC"), ("Japan", "LOC"), ("May 2023", "DATE"), ("United States", "LOC"), ("United Kingdom", "LOC"), ("France", "LOC"), ("Germany", "LOC"), ("Italy", "LOC"), ("Canada", "LOC")]},
    {"text": "Warren Buffett's Berkshire Hathaway, headquartered in Omaha, Nebraska, reported record profits in fiscal year 2022 despite global market volatility.",
     "entities": [("Warren Buffett", "PER"), ("Berkshire Hathaway", "ORG"), ("Omaha", "LOC"), ("Nebraska", "LOC"), ("2022", "DATE")]},
    {"text": "Following negotiations between Russia and Ukraine mediated by Turkey in Istanbul in March 2022, a ceasefire agreement was drafted.",
     "entities": [("Russia", "LOC"), ("Ukraine", "LOC"), ("Turkey", "LOC"), ("Istanbul", "LOC"), ("March 2022", "DATE")]},

    # --- 개체 없거나 매우 희귀한 케이스 ---
    {"text": "The algorithm processes data in real time using parallel execution.",
     "entities": []},
    {"text": "Scientists discovered a new protein involved in cellular respiration.",
     "entities": []},
    {"text": "Quetzalcoatl was worshipped by the Aztec civilization as a feathered serpent deity.",
     "entities": [("Quetzalcoatl", "MISC"), ("Aztec", "MISC")]},
    {"text": "The Atacama Desert in Chile is one of the driest places on Earth.",
     "entities": [("Atacama Desert", "LOC"), ("Chile", "LOC"), ("Earth", "LOC")]},
    {"text": "CERN's Large Hadron Collider detected evidence of the Higgs boson in July 2012.",
     "entities": [("CERN", "ORG"), ("Large Hadron Collider", "MISC"), ("Higgs boson", "MISC"), ("July 2012", "DATE")]},
]

print(f"풀 크기: {len(POOL)}개 문장")
df = pd.DataFrame(POOL)
df['n_entities'] = df['entities'].apply(len)
df['entity_types'] = df['entities'].apply(lambda es: list(set(t for _, t in es)))
print(df[['text', 'n_entities']].head(5).to_string())

---
## 3. 도메인 피처 설계

### 설계 원칙

> **피처가 예시들의 구조적 역할(Role)을 구분해줘야 AISO가 작동합니다.**

Elliptic 실험에서 PCA-30은 실패하고 Dom-12가 성공한 이유가 이겁니다.  
PCA는 데이터를 둥그스름한 덩어리로 뭉갭니다 → M 행렬이 의미 있는 분리를 못 만듦.  
Dom-12는 `illicit_neigh_ratio` 같이 역할을 직접 인코딩하는 피처를 씁니다 → 선명한 경계.

### NER Few-Shot용 피처 7가지

| 피처 | 의미 | 캡처하는 역할 |
|------|------|---------------|
| `length_norm` | 정규화된 문장 길이 | 짧은 예시 vs 긴 복잡 예시 |
| `entity_density` | 개체명 개수 / 토큰 수 | 희소 vs 밀집 어노테이션 |
| `entity_type_diversity` | 고유 타입 수 / 최대 타입 수 | 단일 타입 vs 혼합 타입 |
| `per_ratio` | PER 개체 비율 | 인명 중심 예시 |
| `org_loc_ratio` | ORG+LOC 개체 비율 | 기관/지명 중심 예시 |
| `date_misc_ratio` | DATE+MISC 개체 비율 | 시간/특수 표현 중심 예시 |
| `vocab_richness` | 고유 단어 수 / 전체 단어 수 | 어휘 다양성 |

**핵심**: 마지막 3개 피처(`per_ratio`, `org_loc_ratio`, `date_misc_ratio`)가 핵심입니다.  
이 피처들이 개체명 '역할'을 직접 인코딩합니다 → M 행렬이 의미 있는 repulsion을 만들 수 있음.

In [ ]:
def extract_features(item):
    """하나의 풀 아이템에서 도메인 피처 추출"""
    text = item["text"]
    entities = item["entities"]
    
    tokens = text.split()
    n_tokens = max(len(tokens), 1)
    n_entities = len(entities)
    
    # 타입별 카운트
    type_counts = {"PER": 0, "ORG": 0, "LOC": 0, "DATE": 0, "MISC": 0}
    for _, etype in entities:
        type_counts[etype] = type_counts.get(etype, 0) + 1
    
    unique_types = len(set(t for _, t in entities)) if entities else 0
    max_types = 5  # PER, ORG, LOC, DATE, MISC
    
    unique_words = len(set(w.lower().strip('.,!?;:') for w in tokens))
    
    return {
        "length_norm":          min(n_tokens / 40.0, 1.0),           # 40토큰을 최대로 정규화
        "entity_density":       min(n_entities / n_tokens, 1.0),
        "entity_type_diversity": unique_types / max_types,
        "per_ratio":            type_counts["PER"] / max(n_entities, 1),
        "org_loc_ratio":        (type_counts["ORG"] + type_counts["LOC"]) / max(n_entities, 1),
        "date_misc_ratio":      (type_counts["DATE"] + type_counts["MISC"]) / max(n_entities, 1),
        "vocab_richness":       unique_words / n_tokens,
    }

# 전체 풀에 대해 피처 추출
features_list = [extract_features(item) for item in POOL]
feature_names = list(features_list[0].keys())
X_features = np.array([[f[k] for k in feature_names] for f in features_list])

print("피처 행렬 shape:", X_features.shape)
print("\n피처 통계:")
feat_df = pd.DataFrame(X_features, columns=feature_names)
print(feat_df.describe().round(3).to_string())

In [ ]:
# 피처 공간 시각화 (PCA 2D)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_features)

# 개체 타입 다양성으로 색칠
type_diversities = X_features[:, feature_names.index("entity_type_diversity")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=type_diversities,
                     cmap='RdYlGn', s=80, alpha=0.8, edgecolors='k', linewidths=0.5)
plt.colorbar(sc, ax=axes[0], label='entity_type_diversity')
for i, item in enumerate(POOL):
    axes[0].annotate(str(i), (X_2d[i, 0], X_2d[i, 1]), fontsize=6, ha='center')
axes[0].set_title("풀 피처 공간 (PCA 2D)\n색: 개체 타입 다양성", fontsize=12)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")

# 피처 히트맵
im = axes[1].imshow(X_features.T, aspect='auto', cmap='Blues')
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels(feature_names, fontsize=9)
axes[1].set_xlabel("예시 인덱스")
axes[1].set_title("도메인 피처 히트맵", fontsize=12)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()
print("→ 피처 공간에서 예시들이 다양하게 분포됨을 확인")

---
## 4. AISO 구현

GNN 서브그래프 샘플러 버전을 Few-Shot 선택에 맞게 추상화했습니다.  
핵심 로직은 동일합니다: 에이전트가 피처 공간을 탐색하며 풀 아이템에 방문 카운트를 누적.

In [ ]:
class AISOfewshot:
    """
    AISO 기반 Few-Shot 예시 선택기.
    
    파라미터
    --------
    n_agents : 에이전트 수. 많을수록 커버리지 증가, 속도 감소. 권장: 15~30
    n_iter   : 반복 횟수. 많을수록 수렴, 풀이 작으면 50~100으로 충분
    n_types  : 타입 벡터 차원 K. None이면 피처 차원 D와 동일 (권장)
    alpha    : 이동 스텝 크기. 0.1~0.3 권장
    beta     : 타입 동화율. 0.05~0.15 권장
    m_low    : M 행렬 하한 (음수 → repulsion 생성). -0.5~-1.0 권장
    m_high   : M 행렬 상한 (양수 → attraction). 1.0~2.0 권장
    """
    def __init__(self, n_agents=20, n_iter=80, n_types=None,
                 alpha=0.25, beta=0.08, m_low=-0.5, m_high=2.0, seed=42):
        self.n_agents = n_agents
        self.n_iter   = n_iter
        self.n_types  = n_types
        self.alpha    = alpha
        self.beta     = beta
        self.m_low    = m_low
        self.m_high   = m_high
        self.seed     = seed
        # 실행 후 접근 가능한 내부 상태
        self.visit_counts_ = None
        self.agent_history_ = None

    def _normalize(self, X):
        mn = X.min(axis=0); mx = X.max(axis=0)
        return (X - mn) / (mx - mn + 1e-8)

    def select(self, features: np.ndarray, n_select: int,
               replace: bool = False) -> tuple:
        """
        Parameters
        ----------
        features  : (N_pool, D) 도메인 피처 행렬
        n_select  : 선택할 예시 수
        replace   : 중복 허용 여부

        Returns
        -------
        selected_indices : 선택된 인덱스 배열
        visit_counts     : 각 풀 아이템의 방문 카운트
        """
        rng = np.random.RandomState(self.seed)
        N_pool, D = features.shape
        K = self.n_types if self.n_types is not None else D

        X_norm = self._normalize(features)

        # 초기화
        init_idx = rng.choice(N_pool, self.n_agents, replace=True)
        agent_pos = X_norm[init_idx].copy().astype(float)
        agent_W   = rng.dirichlet(np.ones(K), self.n_agents)     # (n_agents, K)
        M         = rng.uniform(self.m_low, self.m_high, (K, K)) # (K, K)
        visit     = np.zeros(N_pool)
        history   = []  # 시각화용

        for t in range(self.n_iter):
            # 전체 호환성 행렬 한번에 계산: (n_agents, n_agents)
            C = agent_W @ M @ agent_W.T
            np.fill_diagonal(C, 0)

            for i in range(self.n_agents):
                ci = C[i].copy()

                # 상위 3명(끌림) / 하위 3명(밀어냄)
                n_nb = min(3, self.n_agents - 1)
                attract_idx = np.argsort(ci)[-n_nb:]
                repel_idx   = np.argsort(ci)[:n_nb]

                # 합력 계산
                force = np.zeros(D)
                for j in attract_idx:
                    force += ci[j] * (agent_pos[j] - agent_pos[i])
                for j in repel_idx:
                    force += ci[j] * (agent_pos[j] - agent_pos[i])  # ci[j] < 0이면 자동 반전
                force /= (2 * n_nb)

                # 후보 위치 → 가장 가까운 풀 아이템으로 스냅
                candidate = np.clip(agent_pos[i] + self.alpha * force, 0, 1)
                dists = np.linalg.norm(X_norm - candidate, axis=1)
                nearest = int(np.argmin(dists))

                agent_pos[i] = X_norm[nearest]
                visit[nearest] += 1.0

                # 가장 호환성 높은 파트너의 타입 부분 동화
                best_j = int(attract_idx[np.argmax(ci[attract_idx])])
                W_new = (1 - self.beta) * agent_W[i] + self.beta * agent_W[best_j]
                agent_W[i] = W_new / W_new.sum()

            if t % 10 == 0:
                history.append(agent_pos.copy())

        # 방문 빈도에 비례한 확률로 최종 선택
        probs = visit + 1.0  # +1: 한 번도 안 방문한 아이템도 최소 확률
        probs /= probs.sum()
        selected = rng.choice(N_pool, n_select, replace=replace, p=probs)

        self.visit_counts_  = visit
        self.agent_history_ = history
        return selected, visit

print("AISOfewshot 클래스 정의 완료")

---
## 5. AISO 실행 및 랜덤 선택 비교

In [ ]:
N_SELECT = 8  # 선택할 Few-Shot 예시 수

# ── AISO 실행 ──────────────────────────────────────────────────
aiso = AISOfewshot(
    n_agents=20,
    n_iter=100,
    n_types=None,   # None → 피처 차원(7)과 동일
    alpha=0.25,
    beta=0.08,
    m_low=-0.5,
    m_high=2.0,
    seed=42
)
aiso_idx, visit = aiso.select(X_features, N_SELECT)

# ── 랜덤 선택 (비교 기준) ──────────────────────────────────────
rng = np.random.RandomState(42)
random_idx = rng.choice(len(POOL), N_SELECT, replace=False)

print("=" * 60)
print(f"[AISO] 선택된 인덱스: {sorted(aiso_idx)}")
print(f"[Random] 선택된 인덱스: {sorted(random_idx)}")
print("=" * 60)

# ── 선택 결과 출력 ────────────────────────────────────────────
for label, indices in [("AISO", aiso_idx), ("Random", random_idx)]:
    print(f"\n{'─'*60}")
    print(f"[{label}] 선택된 예시")
    print(f"{'─'*60}")
    for rank, idx in enumerate(indices):
        entities_str = ", ".join(f"{e}({t})" for e, t in POOL[idx]["entities"]) or "없음"
        print(f"  {rank+1}. [{idx:02d}] {POOL[idx]['text'][:70]}")
        print(f"       → {entities_str}")

In [ ]:
# ── 다양성 지표 비교 ──────────────────────────────────────────

def compute_diversity_metrics(indices, pool, features):
    """선택된 세트의 다양성 지표 계산"""
    sel_features = features[indices]
    
    # 1. 평균 쌍별 거리 (높을수록 다양)
    pairs = list(combinations(range(len(indices)), 2))
    if pairs:
        pairwise_dists = [np.linalg.norm(sel_features[i] - sel_features[j]) for i, j in pairs]
        avg_pairwise_dist = np.mean(pairwise_dists)
    else:
        avg_pairwise_dist = 0.0
    
    # 2. 커버된 개체 타입 수 (높을수록 다양)
    all_types = set()
    for idx in indices:
        for _, t in pool[idx]["entities"]:
            all_types.add(t)
    n_entity_types = len(all_types)
    
    # 3. 길이 분산 (다양한 길이 커버)
    lengths = [len(pool[idx]["text"].split()) for idx in indices]
    length_std = np.std(lengths)
    
    # 4. 피처 분산 평균
    feat_var = np.mean(np.var(sel_features, axis=0))
    
    return {
        "avg_pairwise_dist": avg_pairwise_dist,
        "entity_types_covered": n_entity_types,
        "length_std": length_std,
        "feature_variance": feat_var,
    }

aiso_metrics   = compute_diversity_metrics(aiso_idx, POOL, X_features)
random_metrics = compute_diversity_metrics(random_idx, POOL, X_features)

# 랜덤 선택 5회 평균으로 공정 비교
random_runs = []
for seed in range(5):
    r_idx = np.random.RandomState(seed).choice(len(POOL), N_SELECT, replace=False)
    random_runs.append(compute_diversity_metrics(r_idx, POOL, X_features))
random_mean = {k: np.mean([r[k] for r in random_runs]) for k in aiso_metrics}

print("\n다양성 지표 비교")
print(f"{'지표':<25} {'AISO':>10} {'Random(평균)':>14}")
print("-" * 52)
for k in aiso_metrics:
    a_val = aiso_metrics[k]
    r_val = random_mean[k]
    better = "✓" if a_val >= r_val else " "
    print(f"{k:<25} {a_val:>10.4f} {r_val:>14.4f}  {better}")
print("\n✓ = AISO가 Random보다 우세")

In [ ]:
# ── 시각화: 피처 공간에서의 선택 분포 ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pca2 = PCA(n_components=2, random_state=42)
X_2d_all = pca2.fit_transform(X_features)

for ax, indices, label, color in [
    (axes[0], aiso_idx, "AISO", "#e74c3c"),
    (axes[1], random_idx, "Random", "#3498db")
]:
    # 전체 풀
    ax.scatter(X_2d_all[:, 0], X_2d_all[:, 1],
               c='#bdc3c7', s=50, alpha=0.6, zorder=1, label='풀 전체')
    # 선택된 예시
    ax.scatter(X_2d_all[indices, 0], X_2d_all[indices, 1],
               c=color, s=200, zorder=3, edgecolors='k', linewidths=1.2,
               label=f'{label} 선택 ({N_SELECT}개)')
    for i, idx in enumerate(indices):
        ax.annotate(str(idx), (X_2d_all[idx, 0], X_2d_all[idx, 1]),
                    fontsize=8, ha='center', va='center', color='white', fontweight='bold', zorder=4)
    ax.set_title(f"{label} 선택\n(avg_dist={compute_diversity_metrics(indices, POOL, X_features)['avg_pairwise_dist']:.3f})",
                 fontsize=12)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("피처 공간에서의 선택 분포 비교", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 방문 카운트 시각화 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(POOL))
bars = ax.bar(x, visit, color='#3498db', alpha=0.7, label='방문 카운트')

# AISO 선택된 예시 강조
for idx in aiso_idx:
    bars[idx].set_color('#e74c3c')
    bars[idx].set_alpha(1.0)

ax.set_xlabel("예시 인덱스")
ax.set_ylabel("방문 횟수")
ax.set_title("AISO 에이전트 방문 카운트\n(빨간색 = 최종 선택된 예시)")
ax.axhline(y=visit.mean(), color='gray', linestyle='--', alpha=0.7, label=f'평균 ({visit.mean():.1f})')

red_patch = mpatches.Patch(color='#e74c3c', label='선택된 예시')
blue_patch = mpatches.Patch(color='#3498db', alpha=0.7, label='미선택')
ax.legend(handles=[red_patch, blue_patch, plt.Line2D([0], [0], color='gray', linestyle='--')],
          labels=['선택된 예시', '미선택', f'평균 ({visit.mean():.1f})'])

plt.tight_layout()
plt.show()
print(f"방문된 고유 예시 수: {(visit > 0).sum()} / {len(POOL)}")

---
## 6. 실제 LLM 스코어 함수 연결 방법

위 AISO는 **방문 빈도 기반** 다양성 선택입니다 (스코어 함수 없음).  
실제로 LLM 성능(BLEU, accuracy 등)으로 **방법 간 비교**를 하려면:

```python
# ─── 여기에 실제 LLM 평가를 연결 ───────────────────────────
def score_fn(selected_indices, pool, eval_examples):
    """
    selected_indices : AISO가 고른 예시 인덱스
    pool             : 전체 풀
    eval_examples    : 평가용 held-out 예시들
    
    Returns: float (높을수록 좋음)
    """
    few_shot_prompt = build_prompt([pool[i] for i in selected_indices])
    predictions = []
    for eval_ex in eval_examples:
        full_prompt = few_shot_prompt + "\n" + eval_ex["input"]
        output = llm.generate(full_prompt)  # 여기를 교체
        predictions.append(output)
    return compute_bleu(predictions, [e["reference"] for e in eval_examples])

# 비교 실행
aiso_score   = score_fn(aiso_idx, POOL, eval_set)
random_score = score_fn(random_idx, POOL, eval_set)
print(f"AISO: {aiso_score:.4f} | Random: {random_score:.4f}")
```

**주의**: 스코어 함수는 AISO 내부 루프에 영향을 주지 않습니다. 선택 후 외부에서 평가합니다.  
AISO 자체는 항상 다양성 기반으로 선택 → 그 다양성이 LLM 성능을 높이는지 외부에서 검증합니다.

In [ ]:
# ── 모의 LLM 평가 (실제 LLM 없이 개념 시연) ──────────────────
# 실제 프로젝트에서는 이 함수를 진짜 LLM 호출로 교체하세요

def mock_llm_score(selected_indices, pool, features, noise_std=0.05, seed=0):
    """
    실제 LLM 평가를 모사하는 함수.
    다양성이 높은 세트일수록 높은 점수를 반환합니다.
    """
    rng = np.random.RandomState(seed)
    sel_feat = features[selected_indices]
    # 다양성 프록시: 평균 쌍별 거리
    pairs = list(combinations(range(len(selected_indices)), 2))
    diversity = np.mean([np.linalg.norm(sel_feat[i] - sel_feat[j]) for i, j in pairs])
    score = 0.5 + 0.4 * diversity + rng.normal(0, noise_std)
    return float(np.clip(score, 0, 1))

# 5시드 반복으로 안정성 비교
n_trials = 5
aiso_scores, random_scores = [], []

for seed in range(n_trials):
    a_score = mock_llm_score(aiso_idx, POOL, X_features, seed=seed)
    r_idx   = np.random.RandomState(seed).choice(len(POOL), N_SELECT, replace=False)
    r_score = mock_llm_score(r_idx, POOL, X_features, seed=seed)
    aiso_scores.append(a_score)
    random_scores.append(r_score)

print("모의 LLM 점수 비교 (5 시드)")
print(f"AISO   : mean={np.mean(aiso_scores):.4f}, std={np.std(aiso_scores):.4f}")
print(f"Random : mean={np.mean(random_scores):.4f}, std={np.std(random_scores):.4f}")
print(f"\n→ 실제 LLM 평가로 교체하면 위 score_fn 패턴 사용")

---
## 7. 다른 태스크에 적용할 때 피처 설계 가이드

### 핵심 원칙
> 피처가 예시의 **구조적 역할(Role)**을 직접 인코딩해야 합니다.  
> 피처 간 차이가 선명할수록 M 행렬이 유효한 inductive bias를 가질 수 있습니다.

### 태스크별 추천 피처

| 태스크 | 피처 예시 | 캡처하는 역할 |
|--------|-----------|---------------|
| **번역** | 문장 길이, 구문 복잡도, 전문 용어 밀도, 숫자/날짜 포함 여부 | 단순 문장 vs 복잡 문장, 기술 텍스트 vs 일반 텍스트 |
| **감성 분류** | 극성 강도, 부정어 밀도, 주관성 점수, 도메인(리뷰/뉴스) | 명확한 감성 vs 중립, 긍정/부정 비율 |
| **요약** | 원문 길이, 압축비, 핵심 문장 위치, 수사 구조 | 짧은 뉴스 vs 긴 보고서, 구조적 글 vs 서술형 |
| **코드 생성** | 복잡도(순환 복잡도), 함수 수, 의존성 수, 도메인(알고리즘/UI/DB) | 간단한 함수 vs 복잡한 시스템 |
| **QA** | 질문 유형(when/who/what/why), 답변 길이, 추론 깊이 | 사실 조회 vs 추론 요구 |

### ❌ 피하는 패턴
- **PCA/임베딩 직접 사용**: 구조를 뭉개서 역할 구분 불가
- **지나치게 많은 피처 (>20)**: M 행렬이 커져서 랜덤 초기화 분산 증가 → lottery 문제 재발
- **중복 피처**: 상관관계 높은 피처들은 M이 같은 방향으로 두 번 당겨서 편향 발생

### ✅ 권장 패턴
- **7~15개 피처**: Dom-12가 최적이었던 것처럼 중간 범위가 안정적
- **역할을 직접 수치화**: `illicit_neigh_ratio` 같은 직관적 피처
- **피처 수 = N_TYPES**: 타입 차원을 피처 차원과 맞추는 것이 기본값 (코드에서 `n_types=None`)

In [ ]:
# ── 파라미터 민감도 빠른 체크 ─────────────────────────────────
# n_select를 바꿔가며 다양성이 어떻게 변하는지 확인

results = []
for n_sel in [4, 6, 8, 10, 12]:
    a = AISOfewshot(n_agents=20, n_iter=100, seed=42)
    idx_a, _ = a.select(X_features, n_sel)
    m_a = compute_diversity_metrics(idx_a, POOL, X_features)

    rand_dists = []
    for s in range(5):
        r_idx = np.random.RandomState(s).choice(len(POOL), n_sel, replace=False)
        rand_dists.append(compute_diversity_metrics(r_idx, POOL, X_features)["avg_pairwise_dist"])

    results.append({
        "n_select": n_sel,
        "AISO_dist": m_a["avg_pairwise_dist"],
        "Random_dist": np.mean(rand_dists),
        "AISO_types": m_a["entity_types_covered"],
    })

res_df = pd.DataFrame(results)
res_df["gain"] = res_df["AISO_dist"] - res_df["Random_dist"]
print(res_df.to_string(index=False))
print("\n→ gain > 0 이면 AISO가 Random보다 다양한 예시를 선택")

---
## 8. 요약 및 체크리스트

### AISO가 잘 작동하는 조건
- [x] 풀 안에 구조적으로 다양한 아이템들이 있음
- [x] 그 다양성을 직접 인코딩하는 도메인 피처를 설계함
- [x] 피처 수가 7~15개 범위 내
- [x] 선택 후 외부 평가로 효과를 검증할 수 있음

### 사용 방법
```python
aiso = AISOfewshot(
    n_agents=20,   # 에이전트 수
    n_iter=100,    # 반복 횟수
    n_types=None,  # None → 피처 차원과 동일
    alpha=0.25,    # 이동 스텝
    beta=0.08,     # 타입 동화율
    m_low=-0.5,    # M 하한 (repulsion 강도)
    m_high=2.0,    # M 상한 (attraction 강도)
    seed=42
)
selected_idx, visit_counts = aiso.select(features_matrix, n_select=8)
```

### 성능이 나쁠 때 체크
1. **피처가 잘 설계됐는가?** → `feat_df.describe()` 확인, 분산이 0에 가까운 피처 제거
2. **n_agents가 충분한가?** → 풀 크기의 1/3 이상 권장
3. **n_iter가 충분한가?** → `visit_counts`에서 0이 많으면 반복 증가
4. **n_types를 조정해봤는가?** → 피처 수와 다르게 설정해서 실험